In [90]:
!pip install groq

In [91]:
!pip install -q groq pandas

In [92]:
from groq import Groq
import pandas as pd
import json

In [93]:
GROQ_API_KEY = "YOUR_GROQ_API_KEY"

client = Groq(api_key=GROQ_API_KEY)

In [94]:
data = {
    "product": ["Laptop", "Phone", "Laptop", "Tablet", "Phone"],
    "quantity": [2, 5, 1, 4, 3],
    "price": [60000, 30000, 60000, 25000, 30000]
}

df = pd.DataFrame(data)

df.to_csv("sales.csv", index=False)

print(df)

  product  quantity  price
0  Laptop         2  60000
1   Phone         5  30000
2  Laptop         1  60000
3  Tablet         4  25000
4   Phone         3  30000


In [95]:
user_goal = input("What would you like me to do? ")

What would you like me to do? give me top 3 products by revenue


In [96]:
planning_prompt = f"""
You are the planning component of a Data Assistant AI Agent.

Break the user's goal into logical steps.

Do not execute any tools.
Do not calculate the answer.

The plan should describe what needs to be done to complete
the user's goal.

User goal:
{user_goal}
"""

In [97]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": planning_prompt
        }
    ]
)

plan = response.choices[0].message.content

print(plan)

**Plan to Deliver the Top 3 Products by Revenue**

1. **Identify the Data Source**
   - Locate the database/table or data file (e.g., `sales_transactions`, `orders`, or `product_sales`) that contains individual sales records with product identifiers and revenue values.
   - Verify that the data includes the necessary dimensions: product ID/name and revenue per transaction.

2. **Confirm the Data Schema**
   - List the relevant columns (e.g., `product_id`, `product_name`, `revenue`, `transaction_date`).
   - Ensure revenue is stored as a numeric type and that currency units are consistent across records.

3. **Data Extraction**
   - Query the data source to retrieve all rows needed for revenue aggregation.
   - If the data volume is large, consider filtering by a relevant time period (e.g., last fiscal year) if appropriate.

4. **Aggregate Revenue by Product**
   - Group the extracted records by `product_id` (and optionally `product_name` for readability).
   - Sum the `revenue` field f

In [98]:
def read_csv():
    df = pd.read_csv("sales.csv")
    return df

In [99]:
def inspect_dataset(df):
    return {
        "columns": list(df.columns),
        "row_count": len(df),
        "data_types": df.dtypes.astype(str).to_dict(),
        "missing_values": df.isnull().sum().to_dict()
    }

In [100]:
def calculate_revenue(df):

    df = df.copy()

    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    df["revenue"] = df["quantity"] * df["price"]

    return df

In [101]:
def top_products(df):

    result = (
        df.groupby("product")["revenue"]
        .sum()
        .reset_index()
        .rename(columns={"revenue": "total_revenue"})
        .sort_values("total_revenue", ascending=False)
        .head(3)
    )

    return result

In [102]:
def save_csv(df):

    output_path = "top_3_products_by_revenue.csv"

    df.to_csv(output_path, index=False)

    return {
        "status": "success",
        "file": output_path,
        "rows_saved": len(df)
    }

In [103]:
def verify_output(file_path):

    df = pd.read_csv(file_path)

    return {
        "file_exists": True,
        "rows": len(df),
        "columns": list(df.columns),
        "data": df.to_dict(orient="records")
    }

In [104]:
tool_registry = {
    "read_csv": read_csv,
    "inspect_dataset": inspect_dataset
}

In [105]:
def convert_plan_to_actions(plan):

    action_prompt = f"""
You are the Action Planner of a Data Assistant AI Agent.

The planner has already created a high-level plan for the user's goal.

Your job is to convert that plan into a sequence of executable
actions that the agent can perform using its available tools.

AVAILABLE TOOLS:

{available_tools}

PLAN:

{plan}

RULES:

1. Convert the plan into executable actions.
2. Use ONLY tools from the AVAILABLE TOOLS list.
3. Do NOT invent tools.
4. Do NOT execute any tool.
5. Do NOT calculate or answer the user's question.
6. Preserve the logical order of the plan.
7. Each action must represent an operation that our agent can actually execute.
8. Include dataset inspection/validation before analysis when required.
9. Do not assume that columns, values, or information exist.
10. Return ALL required actions.
11. Return ONLY valid JSON.

Return the actions in exactly this format:

[
    {{
        "step": 1,
        "action": "tool_name",
        "purpose": "Why this action is needed"
    }}
]
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": action_prompt
            }
        ]
    )

    llm_output = response.choices[0].message.content

    # Remove markdown code fences if Groq adds them
    if llm_output.startswith("```json") and llm_output.endswith("```"):
        llm_output = llm_output[len("```json"):-len("```")].strip()

    try:
        actions = json.loads(llm_output)
        return actions

    except json.JSONDecodeError as e:
        print("JSON Decode Error:", e)
        print("Content that caused the error:", llm_output)
        raise

In [106]:
actions = convert_plan_to_actions(plan)

print(json.dumps(actions, indent=2))

[
  {
    "step": 1,
    "action": "read_csv",
    "purpose": "Load the sales data file into the agent's working memory for analysis."
  },
  {
    "step": 2,
    "action": "inspect_dataset",
    "purpose": "Verify the dataset's structure, column names, data types, and identify any missing values or inconsistencies before performing calculations."
  },
  {
    "step": 3,
    "action": "calculate_revenue",
    "purpose": "Compute the revenue for each sales transaction using the formula quantity \u00d7 price to prepare for revenue aggregation."
  },
  {
    "step": 4,
    "action": "top_products",
    "purpose": "Aggregate the calculated revenue by product, rank products in descending order, and return the top 3 products by total revenue."
  },
  {
    "step": 5,
    "action": "save_csv",
    "purpose": "Persist the top 3 products result to a CSV file for further use or delivery to the user."
  },
  {
    "step": 6,
    "action": "verify_output",
    "purpose": "Read the saved CSV and co

In [107]:
def execute_action(action, state):

    if action == "read_csv":

        result = read_csv()

        state["results"]["dataset"] = result

        return result

    elif action == "inspect_dataset":

        df = state["results"]["dataset"]

        return inspect_dataset(df)

    elif action == "calculate_revenue":

        df = state["results"]["dataset"]

        result = calculate_revenue(df)

        state["results"]["revenue_data"] = result

        return result

    elif action == "top_products":

        df = state["results"]["revenue_data"]

        result = top_products(df)

        state["results"]["top_products"] = result

        return result

    elif action == "save_csv":

        df = state["results"]["top_products"]

        result = save_csv(df)

        state["results"]["output_file"] = result["file"]

        return result

    elif action == "verify_output":

        file_path = state["results"]["output_file"]

        return verify_output(file_path)

    else:

        raise ValueError(f"Unknown action: {action}")

In [108]:
state = {
    "goal": user_goal,
    "plan": plan,
    "actions": actions,
    "completed_actions": [],
    "observations": [],
    "results": {}
}

In [109]:
def update_state(state, action, result):
    state["completed_actions"].append(action)

    state["observations"].append({
        "action": action,
        "result": str(result)
    })

    return state

In [110]:
available_tools = """
Available tools:

1. read_csv
   Reads the sales CSV file.
   No arguments.

2. inspect_dataset
   Inspects the dataset columns, row count,
   data types, and missing values.
   Requires the dataset.

3. calculate_revenue
   Calculates revenue for every sales row.
   Revenue = quantity × price.
   Requires the dataset.

4. top_products
   Calculates total revenue per product
   and returns the top 3 products.
   Requires calculated revenue data.

5. save_csv
   Saves the final result as a CSV file.
   Requires the top products result.

6. verify_output
   Reads the saved CSV and verifies that
   the output contains the expected results.
"""

In [111]:
def decide_next_action(state):

    state_for_llm = {
        "goal": state["goal"],
        "plan": state["plan"],
        "actions": state["actions"],
        "completed_actions": state["completed_actions"],
        "observations": state["observations"]
    }

    decision_prompt = f"""
You are the Decision Maker of a Data Assistant AI Agent.

Your job is to choose the SINGLE NEXT ACTION that should be executed
to complete the user's goal.

The plan has already been created.
The executable actions have already been created.

Do NOT create a new plan.
Do NOT return multiple actions.

AVAILABLE ACTIONS:

{json.dumps(state["actions"], indent=2)}

CURRENT STATE:

{json.dumps(state_for_llm, indent=2, default=str)}

RULES:

1. Choose exactly ONE action.
2. Choose only an action from AVAILABLE ACTIONS.
3. Do not invent an action.
4. Do not execute the action yourself.
5. Do not repeat an action that has already been successfully completed.
6. Consider the observations from previous actions.
7. Make sure the required information for an action is available before
   choosing it.
8. Never invent columns, values, or information.
9. If the available data is insufficient to complete the user's goal,
   choose "request_information".
10. If the user's goal has been successfully completed,
    choose "finish".
11. Return ONLY valid JSON.

Return exactly this format:

{{
    "action": "tool_name",
    "reason": "Short explanation for choosing this action."
}}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": decision_prompt
            }
        ]
    )

    llm_output = response.choices[0].message.content

    # Remove markdown code fences if Groq adds them
    if llm_output.startswith("```json") and llm_output.endswith("```"):
        llm_output = llm_output[len("```json"):-len("```")].strip()

    try:
        decision = json.loads(llm_output)
        return decision

    except json.JSONDecodeError as e:
        print("JSON Decode Error:", e)
        print("Content:", llm_output)
        raise

In [112]:
while True:

    # 1. DECIDE what to do next
    decision = decide_next_action(state)

    action = decision["action"]

    print("\nNext action:", action)
    print("Reason:", decision["reason"])

    # 2. Check whether the agent is finished
    if action == "finish":
        print("\nAgent finished the task.")
        break

    # 3. Execute the selected action
    result = execute_action(action, state)

    print("\nTool executed:", action)
    print("Result:")

    if isinstance(result, pd.DataFrame):
        print(result)
    else:
        print(result)

    # 4. Observe + update state
    state["completed_actions"].append(action)

    state["observations"].append({
        "action": action,
        "result": str(result)
    })


Next action: read_csv
Reason: Need to load the sales data into memory before any inspection or calculations can be performed.

Tool executed: read_csv
Result:
  product  quantity  price
0  Laptop         2  60000
1   Phone         5  30000
2  Laptop         1  60000
3  Tablet         4  25000
4   Phone         3  30000

Next action: inspect_dataset
Reason: The data has been loaded but its structure and column types haven't been verified yet, so we should inspect the dataset before performing calculations.

Tool executed: inspect_dataset
Result:
{'columns': ['product', 'quantity', 'price'], 'row_count': 5, 'data_types': {'product': 'object', 'quantity': 'int64', 'price': 'int64'}, 'missing_values': {'product': 0, 'quantity': 0, 'price': 0}}

Next action: calculate_revenue
Reason: Revenue per transaction has not yet been computed; next logical step.

Tool executed: calculate_revenue
Result:
  product  quantity  price  revenue
0  Laptop         2  60000   120000
1   Phone         5  3000